# 05. FS-Versioned 3D-TransUNet

This notebook runs four short fine-tuning experiments from the base checkpoint produced by `04_train.ipynb`.
All splits use `orig.pt` / `aparc+aseg.pt` tensors (256³, 1 mm isotropic).

- FS v5: train/val/test subsets filtered from `tensors` by FS major version
- FS v6: train/val/test subsets filtered from `tensors` by FS major version
- FS v7: train/val/test subsets filtered from `tensors` by FS major version
- FS v8: `data/fs8/subjects` (orig.mgz + aparc+aseg.mgz) → `data/tensors_gcloud` (orig.pt + aparc+aseg.pt), then split reproducibly and train/evaluate

All outputs are written to unique run directories under:
- `checkpoints_fsv5/`
- `checkpoints_fsv6/`
- `checkpoints_fsv7/`
- `checkpoints_fsv8/`

In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scalesurfer.config import DATA_PATH, DEVICE, SEED, MODULE_PATH
from scalesurfer.plot import plot_region_dice, plot_tissue_dice
from scalesurfer.experiments import (
    build_v8_split_from_root,
    build_versioned_splits_from_existing_split,
    convert_gcloud_mgz_to_tensors,
    load_fs_version_map,
    run_short_finetune_experiment,
    validate_training_versions,
)
from scalesurfer.splits import (
    DEFAULT_NOTEBOOK_SPLIT_MANIFEST_PATH,
    build_or_load_study_split_manifest,
    split_from_manifest,
)

print(f"device={DEVICE} | seed={SEED}")

device=cuda | seed=1337


In [2]:
BASE_PATH = MODULE_PATH.parent

BASE_CHECKPOINT = BASE_PATH / "docs" / "notebooks" / "01_volume" / "checkpoints" / "transunet3d_best.pt"
FS_VERSION_JSON = Path("/home/rph/convolutional_ar/segmentation") / "data" / "fs_version_by_dataset_updated.json"

SPLIT_MANIFEST_PATH = DEFAULT_NOTEBOOK_SPLIT_MANIFEST_PATH  # same manifest as 04_train.ipynb

TENSORS_ROOT = DATA_PATH / "tensors"
MGZ_ROOT = BASE_PATH / "data" / "fs8" / "subjects"
TENSORS_ROOT_V8 = BASE_PATH / "data" / "tensors_gcloud"

SPLIT_RATIOS = (0.8, 0.1, 0.1)
RANDOM_SEED = int(SEED)

# Set True to build/update tensors_gcloud from fs8/subjects (orig.mgz + aparc+aseg.mgz -> .pt).
RUN_CONVERSION = False  # precomputed, toggle to True to re-run conversion
CONVERT_JOBS = -1

TRAIN_OVERRIDES = {
    "epochs": 5,
    "lr_scheduler": "cosine_warmup",
    "warmup_steps": 0,         # auto-adjusted by runner to ~10% of total steps
    "cosine_total_steps": 100,
    "lr": 5e-4
}

if not BASE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing base checkpoint: {BASE_CHECKPOINT}")
if not FS_VERSION_JSON.exists():
    raise FileNotFoundError(f"Missing version map: {FS_VERSION_JSON}")

In [3]:
conversion_summary = None
if RUN_CONVERSION:
    conversion_summary = convert_gcloud_mgz_to_tensors(
        gcloud_root=MGZ_ROOT,
        out_root=TENSORS_ROOT_V8,
        n_jobs=CONVERT_JOBS,
    )
    display(pd.DataFrame([conversion_summary]))
else:
    print("Skipping gcloud conversion (RUN_CONVERSION=False)")


Skipping gcloud conversion (RUN_CONVERSION=False)


In [4]:
fs_major_by_dataset = load_fs_version_map(FS_VERSION_JSON)

# Use the same split manifest as 04_train.ipynb (study-based, orig.pt).
split_manifest = build_or_load_study_split_manifest(
    data_root=TENSORS_ROOT,
    manifest_path=SPLIT_MANIFEST_PATH,
    x_filename="orig.pt",
    y_filename="aparc+aseg.pt",
    force_rebuild=False,
)
base_split = split_from_manifest(split_manifest, split_key="train_notebook_split")

# v5/v6/v7: partition the existing split by FS major (no re-splitting).
# HCP paths (no ds* dataset ID) are treated as FS v5 via default_major_for_unmapped.
versioned_splits = build_versioned_splits_from_existing_split(
    base_split=base_split,
    fs_major_by_dataset=fs_major_by_dataset,
    target_majors=(5, 6, 7),
    default_major_for_unmapped=5,
    error_on_unassigned=True,
)

# Assert exact ID-level equivalence with the original split.
def _norm_paths(paths):
    return {str(Path(p).resolve()) for p in paths}

for split_name in ("train", "val", "test"):
    base_ids = _norm_paths(base_split[f"x_{split_name}"])
    combined_ids = _norm_paths(
        list(versioned_splits[5][f"x_{split_name}"])
        + list(versioned_splits[6][f"x_{split_name}"])
        + list(versioned_splits[7][f"x_{split_name}"])
    )
    if base_ids != combined_ids:
        raise RuntimeError(
            f"Split mismatch for {split_name}: "
            f"base={len(base_ids)} combined={len(combined_ids)} "
            f"overlap={len(base_ids & combined_ids)}"
        )

print("v5/v6/v7 split union matches original training split IDs exactly.")

# v8: all samples in tensors_gcloud are FS v8 outputs.
split_v8 = build_v8_split_from_root(
    tensors_root=TENSORS_ROOT_V8,
    seed=RANDOM_SEED,
    ratios=SPLIT_RATIOS,
)

# Enforce version validity in training sets for v5/v6/v7.
for major in (5, 6, 7):
    _ = validate_training_versions(
        versioned_splits[major]["x_train"],
        fs_major_by_dataset,
        expected_major=major,
        default_major=(5 if major == 5 else None),
    )

# v8 tensors share dataset IDs with v5-v7 data in the version map, so we
# skip the expected_major check and just confirm all paths are accounted for.
print(f"v8 split: n_train={len(split_v8['x_train'])} n_val={len(split_v8['x_val'])} n_test={len(split_v8['x_test'])}")

split_rows = []
for major in (5, 6, 7):
    s = versioned_splits[major]
    split_rows.append({
        "fs_major": major,
        "n_train": len(s["x_train"]),
        "n_val": len(s["x_val"]),
        "n_test": len(s["x_test"]),
        "source": str(SPLIT_MANIFEST_PATH),
    })
split_rows.append({
    "fs_major": 8,
    "n_train": len(split_v8["x_train"]),
    "n_val": len(split_v8["x_val"]),
    "n_test": len(split_v8["x_test"]),
    "source": str(TENSORS_ROOT_V8),
})

split_summary_df = pd.DataFrame(split_rows).sort_values("fs_major").reset_index(drop=True)
display(split_summary_df)

v5/v6/v7 split union matches original training split IDs exactly.
v8 split: n_train=386 n_val=48 n_test=48


,fs_major,n_train,n_val,n_test,source
0,5,720,90,90,/home/rph/scalesurfer/docs/notebooks/01_volume...
1,6,1592,202,202,/home/rph/scalesurfer/docs/notebooks/01_volume...
2,7,1077,134,134,/home/rph/scalesurfer/docs/notebooks/01_volume...
3,8,386,48,48,/home/rph/scalesurfer/data/tensors_gcloud


In [5]:
import hashlib

def _norm_set(paths):
    return {str(Path(p).resolve()) for p in paths}

def _fingerprint(paths):
    payload = "\n".join(sorted(_norm_set(paths)))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

current_split = {
    "x_train": list(versioned_splits[5]["x_train"]) + list(versioned_splits[6]["x_train"]) + list(versioned_splits[7]["x_train"]),
    "y_train": list(versioned_splits[5]["y_train"]) + list(versioned_splits[6]["y_train"]) + list(versioned_splits[7]["y_train"]),
    "x_val": list(versioned_splits[5]["x_val"]) + list(versioned_splits[6]["x_val"]) + list(versioned_splits[7]["x_val"]),
    "y_val": list(versioned_splits[5]["y_val"]) + list(versioned_splits[6]["y_val"]) + list(versioned_splits[7]["y_val"]),
    "x_test": list(versioned_splits[5]["x_test"]) + list(versioned_splits[6]["x_test"]) + list(versioned_splits[7]["x_test"]),
    "y_test": list(versioned_splits[5]["y_test"]) + list(versioned_splits[6]["y_test"]) + list(versioned_splits[7]["y_test"]),
}

for key in ("x_train", "y_train", "x_val", "y_val", "x_test", "y_test"):
    cur = _norm_set(current_split[key])
    ref = _norm_set(base_split[key])
    if cur != ref:
        only_ref = sorted(ref - cur)
        only_cur = sorted(cur - ref)
        raise AssertionError(
            f"{key} mismatch: ref_only={len(only_ref)} cur_only={len(only_cur)}\n"
            f"ref_only_example={only_ref[:1]}\n"
            f"cur_only_example={only_cur[:1]}"
        )

print("Exact filename parity OK (x/y train/val/test) vs 04_train.ipynb split manifest.")
print("fingerprints:")
for key in ("x_train", "x_val", "x_test"):
    print(f"  {key}: {_fingerprint(current_split[key])}")

Exact filename parity OK (x/y train/val/test) vs 04_train.ipynb split manifest.
fingerprints:
  x_train: 0c355ad340dc16da
  x_val: b34a4d538e8de37d
  x_test: 9c90876d02042986


In [6]:
# Validate split correctness against manifest counts.
manifest_counts = split_manifest["counts"]["train_notebook_split"]
expected_n_train = manifest_counts["n_train"]
expected_n_val   = manifest_counts["n_val"]
expected_n_test  = manifest_counts["n_test"]

# 1) Raw split IDs (before cache filtering) must match the manifest totals.
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_train"].sum() == expected_n_train, \
    f"train count mismatch: got {split_summary_df[split_summary_df['fs_major'].isin([5, 6, 7])]['n_train'].sum()}, expected {expected_n_train}"
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_val"].sum() == expected_n_val, \
    f"val count mismatch"
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_test"].sum() == expected_n_test, \
    f"test count mismatch"

# 2) Usable counts after cache filtering.
from scalesurfer.config import build_runtime_cfgs
from scalesurfer.api import prepare_data_pipeline

combined_split = {
    "x_train_files": list(versioned_splits[5]["x_train"]) + list(versioned_splits[6]["x_train"]) + list(versioned_splits[7]["x_train"]),
    "y_train_files": list(versioned_splits[5]["y_train"]) + list(versioned_splits[6]["y_train"]) + list(versioned_splits[7]["y_train"]),
    "x_val_files": list(versioned_splits[5]["x_val"]) + list(versioned_splits[6]["x_val"]) + list(versioned_splits[7]["x_val"]),
    "y_val_files": list(versioned_splits[5]["y_val"]) + list(versioned_splits[6]["y_val"]) + list(versioned_splits[7]["y_val"]),
    "x_test_files": list(versioned_splits[5]["x_test"]) + list(versioned_splits[6]["x_test"]) + list(versioned_splits[7]["x_test"]),
    "y_test_files": list(versioned_splits[5]["y_test"]) + list(versioned_splits[6]["y_test"]) + list(versioned_splits[7]["y_test"]),
    "group_split_enabled": False,
}

DATA_CFG_CHECK, _, TRAIN_CFG_CHECK, _ = build_runtime_cfgs(
    data_overrides=combined_split,
    model_overrides={},
    train_overrides={"num_workers": 0},
)
pipeline_check = prepare_data_pipeline(DATA_CFG_CHECK, TRAIN_CFG_CHECK)

n_train_usable = len(pipeline_check.train_ds)
n_val_usable   = len(pipeline_check.val_ds)
n_test_usable  = len(pipeline_check.test_ds) if pipeline_check.test_ds is not None else 0

print(
    f"split validation passed: "
    f"raw={expected_n_train}/{expected_n_val}/{expected_n_test}, "
    f"usable(after cache)={n_train_usable}/{n_val_usable}/{n_test_usable}"
)

split validation passed: raw=3389/426/426, usable(after cache)=3388/426/426


In [7]:
experiment_plan = {
    #5: {"split": versioned_splits[5], "checkpoint_base_dir": Path("checkpoints_fsv5")},
    #6: {"split": versioned_splits[6], "checkpoint_base_dir": Path("checkpoints_fsv6")},
    7: {"split": versioned_splits[7], "checkpoint_base_dir": Path("checkpoints_fsv7")},
    #8: {"split": split_v8,            "checkpoint_base_dir": Path("checkpoints_fsv8")},
}
TRAIN_OVERRIDES["epochs"] = 25
results = {}
for fs_major, spec in experiment_plan.items():

    # if fs_major == 8:
    #     # train longer, this fs version made large changes to aparc+aseg
    #     TRAIN_OVERRIDES["epochs"] = 25

    split = spec["split"]
    if len(split["x_train"]) == 0 or len(split["x_test"]) == 0:
        print(f"[skip fsv{fs_major}] need non-empty train and test split")
        continue

    result = run_short_finetune_experiment(
        run_name=f"fsv{fs_major}",
        fs_major=fs_major,
        split=split,
        base_checkpoint_path=BASE_CHECKPOINT,
        checkpoint_base_dir=spec["checkpoint_base_dir"],
        fs_major_by_dataset=fs_major_by_dataset,
        default_major_for_unmapped=(5 if fs_major == 5 else None),
        train_overrides=TRAIN_OVERRIDES,
        seed=RANDOM_SEED,
        device=DEVICE,
    )
    results[fs_major] = result

summary_rows = []
for fs_major, result in sorted(results.items()):
    summary_rows.append({
        "fs_major": fs_major,
        "run_name": result.run_name,
        "n_train": result.n_train,
        "n_val": result.n_val,
        "n_test": result.n_test,
        "warmup_steps": result.warmup_steps,
        "cosine_total_steps": result.cosine_total_steps,
        "best_epoch": result.best_epoch,
        "best_val_ce": result.best_val_ce,
        "final_test_ce": result.final_test_ce,
        "run_dir": str(result.run_dir),
        "checkpoint_path": str(result.checkpoint_path),
    })

results_summary_df = pd.DataFrame(summary_rows).sort_values("fs_major").reset_index(drop=True)
display(results_summary_df)


fsv7 | epoch 001 train:   0%|          | 0/1077 [00:00<?, ?it/s]

fsv7 | epoch 001 val:   0%|          | 0/134 [00:00<?, ?it/s]

[fsv7] epoch 001 | train_ce=0.0410 | val_ce=0.0473 | lr=2.01e-04 | mb=1 | pc=192 | vram=0.1/25.1GB | t=833.3s
[fsv7] saved best checkpoint -> checkpoints_fsv7/fsv7_20260422_022504/transunet3d_best.pt (epoch=1, val_ce=0.0473)


fsv7 | epoch 002 train:   0%|          | 0/1077 [00:00<?, ?it/s]

fsv7 | epoch 002 val:   0%|          | 0/134 [00:00<?, ?it/s]

...

fsv7 | epoch 025 val:   0%|          | 0/134 [00:00<?, ?it/s]

[fsv7] epoch 025 | train_ce=0.0352 | val_ce=0.0470 | lr=1.00e-06 | mb=1 | pc=192 | vram=0.1/25.1GB | t=725.4s


fsv7 | final test:   0%|          | 0/134 [00:00<?, ?it/s]

test metrics (fast):   0%|          | 0/134 [00:00<?, ?it/s]

,fs_major,run_name,n_train,n_val,n_test,warmup_steps,cosine_total_steps,best_epoch,best_val_ce,final_test_ce,run_dir,checkpoint_path
0,7,fsv7,1077,134,134,2692,26925,15,0.043249,0.047176,checkpoints_fsv7/fsv7_20260422_022504,checkpoints_fsv7/fsv7_20260422_022504/transune...


In [ ]:
# if not results:
#     raise RuntimeError("No completed experiments found.")

# majors = sorted(results.keys())
# fig, axes = plt.subplots(
#     nrows=2,
#     ncols=len(majors),
#     figsize=(7 * len(majors), 14),
#     constrained_layout=True,
# )
# if len(majors) == 1:
#     axes = axes.reshape(2, 1)

# for col, fs_major in enumerate(majors):
#     bundle = results[fs_major].plot_bundle

#     ax_region = axes[0, col]
#     plot_region_dice(bundle, ax=ax_region)
#     ax_region.set_title(f"FS v{fs_major}: Region Dice")

#     ax_tissue = axes[1, col]
#     plot_tissue_dice(bundle, ax=ax_tissue)
#     ax_tissue.set_title(f"FS v{fs_major}: Tissue Dice")

# plot_root = Path("comparison_plots") / time.strftime("%Y%m%d_%H%M%S")
# plot_root.mkdir(parents=True, exist_ok=False)
# fig_path = plot_root / "fsversion_region_tissue_compare.png"
# fig.savefig(fig_path, dpi=200)

# print(f"saved comparison figure -> {fig_path}")
# results_summary_df.to_csv(plot_root / "results_summary.csv", index=False)
# print(f"saved results summary -> {plot_root / 'results_summary.csv'}")

## Notes

- The fine-tuning runner always creates a unique run directory and never overwrites prior runs.
- Each run writes `history.csv`, `sample_metrics.csv`, `region_metrics.csv`, `timing.csv`, and `summary.csv`.
- Region/tissue plots are generated with `plot_region_dice` and `plot_tissue_dice` from `segmentation/plot.py`.
